# Integrating with OpenAI Chat Completions

This guide adds Immune to applications built directly on the `openai` SDK. Each section shows the code you have today,
then the same code with Immune, so you can see exactly what changes.

| You have | Section |
| --- | --- |
| A chat assistant with conversation history | [1](#1.-A-multi-turn-assistant) |
| One client to protect, not the whole process | [2](#2.-Choose-how-to-activate-Immune) |
| Streaming replies | [3](#3.-Streaming) |
| `AsyncOpenAI` and concurrent requests | [4](#4.-Async-and-concurrency) |
| A function-calling loop | [5](#5.-Function-calling) |
| Structured outputs with Pydantic | [6](#6.-Structured-outputs) |
| The Responses API | [7](#7.-The-Responses-API) |
| Code that should raise instead of receiving a safe reply | [8](#8.-When-Immune-blocks:-respond-or-raise) |

**Requirements:** `pip install immune-ai openai`, with `OPENAI_API_KEY` and `TYPESAFE_API_KEY` set.

## 1. A multi-turn assistant

**Before.** A support assistant that keeps the conversation history, as most chat apps do:

In [1]:
from openai import OpenAI

client = OpenAI()
SYSTEM_PROMPT = (
    "You are the support assistant for Pinecrest Outfitters, an online camping and hiking store. "
    "Two-person tents in stock: Ridgeline 2P ($249, 1.9 kg) and Summit Lite 2P ($329, 1.3 kg)."
)


def answer(history: list[dict], question: str) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, *history, {"role": "user", "content": question}]
    response = client.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    return response.choices[0].message.content


history: list[dict] = []
for question in ["Do you sell two-person tents?", "Which one is lighter?"]:
    reply = answer(history, question)
    history += [{"role": "user", "content": question}, {"role": "assistant", "content": reply}]
    print(f"customer:  {question}\nassistant: {reply}\n")

customer:  Do you sell two-person tents?
assistant: Yes — we have two two-person tents in stock:

- **Ridgeline 2P** — **$249**, **1.9 kg**
- **Summit Lite 2P** — **$329**, **1.3 kg**

If you want, I can help you compare them or recommend one based on weight and budget.



customer:  Which one is lighter?
assistant: The **Summit Lite 2P** is lighter at **1.3 kg**. The **Ridgeline 2P** weighs **1.9 kg**.



**After.** Three additions: `immune.init()` at startup, and two context managers that tell Immune which part of the
app this is (`site`) and which conversation (`session`). Sessions let Immune remember, across calls, what untrusted
content a conversation has read.

```diff
+ import immune
  from openai import OpenAI

+ immune.init()
  client = OpenAI()
  ...
+ with immune.site("support-chat"), immune.session("customer-1042"):
      reply = answer(history, question)
```

In [2]:
from openai import OpenAI

import immune

immune.init()
client = OpenAI()
SYSTEM_PROMPT = (
    "You are the support assistant for Pinecrest Outfitters, an online camping and hiking store. "
    "Two-person tents in stock: Ridgeline 2P ($249, 1.9 kg) and Summit Lite 2P ($329, 1.3 kg)."
)


def answer(history: list[dict], question: str) -> tuple[str, immune.Verdict]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, *history, {"role": "user", "content": question}]
    response = client.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    return response.choices[0].message.content, immune.verdict(response)


history: list[dict] = []
for question in ["Do you sell two-person tents?", "Which one is lighter?"]:
    with immune.site("support-chat"), immune.session("customer-1042"):
        reply, verdict = answer(history, question)
    history += [{"role": "user", "content": question}, {"role": "assistant", "content": reply}]
    print(f"customer:  {question}\nassistant: {reply}")
    print(f"verdict:   {verdict.action} · site={verdict.site} · session={verdict.session_id}\n")

customer:  Do you sell two-person tents?
assistant: Yes — we have two-person tents in stock:

- Ridgeline 2P — $249, 1.9 kg
- Summit Lite 2P — $329, 1.3 kg

If you’d like, I can also help compare them by weight or price.
verdict:   allow · site=support-chat · session=customer-1042



customer:  Which one is lighter?
assistant: The **Summit Lite 2P** is lighter at **1.3 kg**.

The **Ridgeline 2P** weighs **1.9 kg**.
verdict:   allow · site=support-chat · session=customer-1042



**What you see:** the same conversation, with a verdict per call, each tagged with the site and session.

> **Tip:** Without `immune.session(...)`, Immune derives a session from the transcript itself, so conversations still
> stay separate. Pass your own id (a user or conversation id) when you have one; it also groups calls in your logs.

## 2. Choose how to activate Immune

| You want to… | Use |
| --- | --- |
| Protect every LLM call in the process (recommended) | `immune.init()` at startup |
| Protect one client object only | `client = immune.protect(OpenAI())` |
| Protect a block of code, such as a script or a test | `with immune.protected(): ...` |

`immune.protect()` keeps the client's own settings (proxies, timeouts, limits and event hooks):

In [3]:
import httpx2
from openai import OpenAI

import immune

client = immune.protect(OpenAI(timeout=30, http_client=httpx2.Client(timeout=30)))

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "In one sentence: what should I pack for a summer day hike?"}],
)
print(response.choices[0].message.content)
print("verdict:", immune.verdict(response).action)

Pack plenty of water, a light snack, sunscreen, a hat, comfortable hiking shoes, a map or phone with GPS, and a light rain layer.
verdict: allow


In [4]:
from openai import OpenAI

import immune

with immune.protected():
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": "In one sentence: how do I dry a wet sleeping bag?"}],
    )
    print(response.choices[0].message.content)
    print("verdict:", immune.verdict(response).action)

Spread it out in a warm, breezy place out of direct sun, and if it’s machine-dryable, tumble on low heat with a couple clean tennis balls or dryer balls, checking often until fully dry.
verdict: allow


## 3. Streaming

**Before.**

In [5]:
from openai import OpenAI

client = OpenAI()
stream = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Give me three tips for choosing hiking boots."}],
    stream=True,
)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

Here

 are

 three

 good

 tips

...

 and

 hiking

 shoes

**

.

**After.** Nothing changes in the loop. Keep the last chunk if you want the verdict: `immune.verdict(chunk)` finds it
by the response id once the stream is finished.

In [6]:
from openai import OpenAI

import immune

immune.init()
client = OpenAI()

stream = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Give me three tips for choosing hiking boots."}],
    stream=True,
)
last_chunk = None
for chunk in stream:
    last_chunk = chunk
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

print("\n\nverdict:", immune.verdict(last_chunk).action)

Sure—here are three useful 

tips for choosing hiking boots:



1. **Match the boot to 

the terrain**
   - Light trails: 

low-cut hiking shoes may 

...

hot, dry weather, breathable boots 

may be 

more comfortable.

If 

you want, I can also give you tips based on **beginner hikers**, **long-distance trekking**, or **winter hiking**.



verdict: allow


**How streaming is screened:** text is released as it arrives, keeping a short tail back. If a reflex finds a
candidate (a link carrying data, a key-shaped string), the rest of the reply waits for Jev's decision and is released
with any redaction applied. Sites where a whole-reply check could act are buffered automatically. To always buffer,
set `streaming: buffered`.

## 4. Async and concurrency

`AsyncOpenAI` works unchanged, and every concurrent call gets its own verdict. In a notebook you can `await` at the
top level; in a script, call `asyncio.run(main())`.

In [7]:
import asyncio

from openai import AsyncOpenAI

import immune

immune.init()
client = AsyncOpenAI()
QUESTIONS = [
    "Is the Ridgeline 2P tent waterproof?",
    "Can I return boots I have worn outside?",
    "Do you ship to Canada?",
]


async def ask(question: str) -> tuple[str, str]:
    response = await client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {
                "role": "system",
                "content": "You are the support assistant for Pinecrest Outfitters. Answer in one sentence.",
            },
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content, immune.verdict(response).action


async def main() -> None:
    results = await asyncio.gather(*(ask(question) for question in QUESTIONS))
    for question, (reply, action) in zip(QUESTIONS, results, strict=True):
        print(f"[{action}] {question}\n        {reply}")


await main()  # in a script: asyncio.run(main())

[allow] Is the Ridgeline 2P tent waterproof?
        Yes, the Ridgeline 2P tent is waterproof with a water-resistant rainfly and sealed seams.
[allow] Can I return boots I have worn outside?
        Yes—boots worn outside can usually be returned only if they’re unworn and in original condition, so if yours have visible wear we may not be able to accept them.
[allow] Do you ship to Canada?
        Yes, we ship to Canada.


> **Tip:** In concurrent code, always pass the response to `immune.verdict(response)`. Calling `immune.verdict()`
> with no argument returns the most recent verdict in the current context, which is only reliable for sequential code.

## 5. Function calling

**Before.** A support agent that can look up orders and issue refunds, with the standard tool-calling loop:

In [8]:
import json

from openai import OpenAI

client = OpenAI()
ORDERS = {"5521": {"item": "Ridgeline 2P tent", "total": 249.00, "status": "delivered"}}
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": "Look up an order by its number.",
            "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "issue_refund",
            "description": "Refund an order, in full or in part.",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "string"}, "amount": {"type": "number"}},
                "required": ["order_id", "amount"],
            },
        },
    },
]


def lookup_order(order_id: str) -> dict:
    return ORDERS.get(order_id, {"error": "no such order"})


def issue_refund(order_id: str, amount: float) -> dict:
    return {"order_id": order_id, "refunded": amount}


FUNCTIONS = {"lookup_order": lookup_order, "issue_refund": issue_refund}


def run_agent(question: str) -> str:
    messages = [
        {"role": "system", "content": "You are the support agent for Pinecrest Outfitters. Use the tools to help."},
        {"role": "user", "content": question},
    ]
    while True:
        response = client.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=TOOLS)
        message = response.choices[0].message
        if not message.tool_calls:
            return message.content
        messages.append(
            {
                "role": "assistant",
                "content": message.content,
                "tool_calls": [c.model_dump() for c in message.tool_calls],
            }
        )
        for call in message.tool_calls:
            result = FUNCTIONS[call.function.name](**json.loads(call.function.arguments))
            print(f"  tool: {call.function.name}({call.function.arguments}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})


print(run_agent("My tent from order 5521 arrived with a broken pole. Please refund it."))

  tool: lookup_order({"order_id":"5521"}) -> {'item': 'Ridgeline 2P tent', 'total': 249.0, 'status': 'delivered'}


  tool: issue_refund({"order_id":"5521","amount":249}) -> {'order_id': '5521', 'refunded': 249}


Done — I’ve refunded $249.00 for order 5521.


**After.** Tell Immune what your tools can do, so it knows which calls send data out, change state or can't be undone.
The loop itself is unchanged; the only other addition collects the verdict of each model call.

```diff
+ immune.init(config={"sites": {"support-agent": {"user_facing": True, "tools": {
+     "issue_refund": {"writes_state": True, "irreversible": True}}}}})
  ...
+ with immune.site("support-agent"):
      print(run_agent(...))
```

In [9]:
import json

from openai import OpenAI

import immune

immune.init(
    config={
        "sites": {
            "support-agent": {
                "user_facing": True,
                "tools": {"issue_refund": {"writes_state": True, "irreversible": True}},
            }
        }
    }
)
client = OpenAI()
ORDERS = {"5521": {"item": "Ridgeline 2P tent", "total": 249.00, "status": "delivered"}}
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": "Look up an order by its number.",
            "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "issue_refund",
            "description": "Refund an order, in full or in part.",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "string"}, "amount": {"type": "number"}},
                "required": ["order_id", "amount"],
            },
        },
    },
]


def lookup_order(order_id: str) -> dict:
    return ORDERS.get(order_id, {"error": "no such order"})


def issue_refund(order_id: str, amount: float) -> dict:
    return {"order_id": order_id, "refunded": amount}


FUNCTIONS = {"lookup_order": lookup_order, "issue_refund": issue_refund}


def run_agent(question: str) -> str:
    messages = [
        {"role": "system", "content": "You are the support agent for Pinecrest Outfitters. Use the tools to help."},
        {"role": "user", "content": question},
    ]
    while True:
        response = client.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=TOOLS)
        verdict = immune.verdict(response)
        print(f"  model call: {verdict.action}  {verdict.explanation}")
        message = response.choices[0].message
        if not message.tool_calls:
            return message.content
        messages.append(
            {
                "role": "assistant",
                "content": message.content,
                "tool_calls": [c.model_dump() for c in message.tool_calls],
            }
        )
        for call in message.tool_calls:
            result = FUNCTIONS[call.function.name](**json.loads(call.function.arguments))
            print(f"  tool: {call.function.name}({call.function.arguments}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})


with immune.site("support-agent"):
    print(run_agent("My tent from order 5521 arrived with a broken pole. Please refund it."))

  model call: allow  no threats detected
  tool: lookup_order({"order_id":"5521"}) -> {'item': 'Ridgeline 2P tent', 'total': 249.0, 'status': 'delivered'}


  model call: allow  observed (would neutralize): data.impersonation
  tool: issue_refund({"order_id":"5521","amount":249}) -> {'order_id': '5521', 'refunded': 249}


  model call: allow  observed (would rewrite): business.ungrounded_number, data.impersonation, output.follows_data_instructions
Done — I’ve refunded the full $249.00 for order 5521.


**What you see:** the same refund, with a verdict for each of the three model calls.

- **The refund went ahead.** After the order lookup, the session has read outside content (the tool result), and
  `issue_refund` can't be undone, so floor rule F10 checked the call: it asked Jev whether the customer asked for the
  refund, Jev said yes, and nothing was held. Had the refund been suggested by something the agent *read* instead,
  Immune would have asked the customer to confirm first.
- **Some detectors were observed.** Detectors outside the floor may flag ordinary traffic while they learn your app,
  such as a refund confirmation read as an unauthorized commitment. They only record `would_action`; that is why they
  start observed. Report a false alarm with `immune.feedback(trace_id, "false_positive")`, and enforce a detector
  per site only once its verdicts fit your traffic (see [Modes, sites and sessions](07_modes_sites_and_sessions.ipynb)).

[Tools and agents](10_tools_and_agents.ipynb) covers confirmations, destinations and loops in depth.

## 6. Structured outputs

Structured outputs work unchanged. For enum and boolean fields, Immune also reads the input independently and records
how confident it is in each field (*schema echo*), in `verdict.echo`.

In [10]:
from typing import Literal

from openai import OpenAI
from pydantic import BaseModel

import immune

immune.init()
client = OpenAI()


class TicketTriage(BaseModel):
    category: Literal["shipping", "returns", "product", "billing", "other"]
    priority: Literal["low", "normal", "urgent"]
    refund_requested: bool


email = (
    "Hi, my order #5521 arrived yesterday but the tent pole is snapped. I leave for a trip on Friday - can you help?"
)
with immune.site("ticket-triage"):
    completion = client.chat.completions.parse(
        model="gpt-5.4-mini",
        messages=[{"role": "system", "content": "Triage customer emails."}, {"role": "user", "content": email}],
        response_format=TicketTriage,
    )
ticket = completion.choices[0].message.parsed
print(ticket)

verdict = immune.verdict(completion)
print("\nverdict:", verdict.action)
for field, distribution in verdict.echo.items():
    best = max(distribution, key=distribution.get)
    print(f"  Immune's own reading of {field}: {best} ({distribution[best]:.2f})")

category='product' priority='urgent' refund_requested=False

verdict: allow
  Immune's own reading of category: product (0.93)
  Immune's own reading of priority: urgent (0.84)
  Immune's own reading of refund_requested: false (0.89)


If the model's field disagrees with Immune's independent reading, `output.echo_disagreement` records it, and on sites
that set `echo.enforce: true` your code gets a refusal instead of a wrong value.

## 7. The Responses API

`client.responses.create(...)` is screened the same way, including `previous_response_id` chains:

In [11]:
from openai import OpenAI

import immune

immune.init()
client = OpenAI()

first = client.responses.create(
    model="gpt-5.4-mini",
    instructions="You are the support assistant for Pinecrest Outfitters.",
    input="What sizes does the Summit Lite sleeping bag come in? Answer briefly.",
)
second = client.responses.create(
    model="gpt-5.4-mini",
    previous_response_id=first.id,
    input="And which one fits someone who is 6 feet tall?",
)
print(first.output_text, "\n")
print(second.output_text)
print("\nsame session:", immune.verdict(first).session_id == immune.verdict(second).session_id)

The Summit Lite sleeping bag comes in **regular** and **long** sizes. 

The **regular** size should fit someone who is **6 feet tall**.

same session: True


## 8. When Immune blocks: respond or raise

By default a blocked call returns a normal response with a safe message (`on_block: respond`), so your UI keeps
working. If your code should handle blocks explicitly, set `on_block: raise` and catch `immune.Blocked`. For OpenAI
clients the exception is also an `openai.OpenAIError`, so existing `except openai.OpenAIError` handling still applies.

In [12]:
import openai
from openai import OpenAI

import immune

immune.init(config={"on_block": "raise", "sites": {"support-chat": {"enforce": ["input.off_task"]}}})
client = OpenAI()

try:
    with immune.site("support-chat"):
        client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[
                {
                    "role": "system",
                    "content": "You are the support assistant for Pinecrest Outfitters, a camping store.",
                },
                {"role": "user", "content": "Write a cover letter for my job application."},
            ],
        )
except immune.Blocked as error:
    print("blocked:", error.verdict.action, "-", error.verdict.explanation)
    print("also an openai.OpenAIError:", isinstance(error, openai.OpenAIError))

blocked: redirect - redirect: input.off_task
also an openai.OpenAIError: True


## Common mistakes

| Avoid | Prefer instead | Why |
| --- | --- | --- |
| Calling `immune.init()` for every request | Once, at startup | Each call rebuilds Immune's runtime |
| `immune.verdict()` in concurrent code | `immune.verdict(response)` | Without an argument you get the latest verdict in this context |
| Pasting retrieved documents into the **system** prompt | Tool results, or `immune.untrusted(text)` in the user message | The system prompt is trusted; documents are not |
| Leaving sites unnamed when the system prompt contains per-request text | `with immune.site("name"):` | Immune can't tell parts of your app apart otherwise |
| Enforcing every detector on day one | Start in `auto`, read `would_action`, then enforce per site | Observe first; enforce what fits your traffic |

## Recap

- `immune.init()` covers sync, async, streaming, tools, structured outputs and the Responses API
- `immune.site()` and `immune.session()` give Immune the context a real app has
- Declare tool capabilities so Immune knows which calls matter
- `on_block` chooses between a safe reply and an exception

Next: [LangChain](04_langchain.ipynb) · [LangGraph](05_langgraph.ipynb) · [Reading verdicts](06_reading_verdicts.ipynb)